# Python 101 - Solutions
## Chapter VIII

---

**For teaching assistants.** This notebook mirrors the exercises in
`../python101_08.ipynb` one for one. Most solutions end with an `assert`, so
running the whole notebook top to bottom is also a self-test: if it runs clean,
every solution still works.

There is usually more than one right answer - if a student's version passes the
same `assert`, it is correct.

In [ ]:
# Run from the chapter folder, so that `helpers`, `./data/...` and `./pics/...`
# resolve exactly the way they do in the lecture notebooks.
import os
import sys

if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print('working directory:', os.getcwd())

In [ ]:
import datetime
import random

import requests
from bs4 import BeautifulSoup

BASE_URI = './data/'

USER_AGENTS = [
    'Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/51.0.2704.64 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/47.0.2526.111 Safari/537.36',
    'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:15.0) Gecko/20100101 Firefox/15.0.1',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2) AppleWebKit/601.3.9 (KHTML, like Gecko) Version/9.0.2 Safari/601.3.9',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/42.0.2311.135 Safari/537.36 Edge/12.246',
    'Mozilla/5.0 (PlayStation 4 3.11) AppleWebKit/537.73 (KHTML, like Gecko)',
]


def get_header(agents):
    return {'User-agent': random.choice(agents)}

### Exercise 1. Articles about Soros on portfolio.hu

The notebook's warning - *"you may find `<article>` tags in weird places that you do not want to include"* - is the whole exercise. A plain `find_all('article')` returns **48** tags, but only **20** of them are search results; the rest are promo boxes for Portfolio Signature, the podcast, penzcentrum.hu and so on.

Print the classes of the tags you got and the right container jumps out: the results carry `category-list-article`. 20 is also exactly the number the notebook says you should see.

In [ ]:
today = datetime.date.today().isoformat()
base_url = 'https://portfolio.hu'
sub_url = '/kereses'
query = {'q': 'Soros', 'df': '1999-02-09', 'dt': today, 'page': 1}

response = requests.get(url=base_url + sub_url, params=query, headers=get_header(USER_AGENTS))
response.raise_for_status()
soup = BeautifulSoup(response.content, 'html.parser')

# step 1: look at what find_all('article') actually gave you
everything = soup.find_all('article')
print('all <article> tags:', len(everything))
for classes in {' '.join(tag.get('class') or [])[:60] for tag in everything}:
    print('   ', classes)

# step 2: keep only the search results
results = soup.select('article.category-list-article')
urls = [tag.find('a').get('href') for tag in results if tag.find('a')]

print(f'\nsearch results: {len(results)}')
for url in urls[:5]:
    print('  ', url)

assert len(everything) > len(results), 'the promo boxes should have been filtered out'
assert len(urls) == len(results)
assert all('portfolio.hu' in url for url in urls)

### Exercise 1.b Only 20 results - so query day by day

One request per day from 1999 to today is over 9700 requests, which is both slow and rude. The function is the deliverable; run it over a short range to show it works, and talk about `time.sleep()` and rate limiting rather than actually hammering the site from 20 laptops at once.

In [ ]:
def search_one_day(term, day):
    """Return the article urls published on a single day for `term`."""
    query = {
        'q': term,
        'df': f'{day.year}-{day.month:0>2}-{day.day:0>2}',
        'dt': f'{day.year}-{day.month:0>2}-{day.day:0>2}',
        'page': 1,
    }
    response = requests.get(base_url + sub_url, params=query,
                            headers=get_header(USER_AGENTS))
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')
    return [tag.find('a').get('href')
            for tag in soup.select('article.category-list-article')
            if tag.find('a')]


one_day = datetime.timedelta(days=1)
day = datetime.date.today() - 7 * one_day

collected = {}
while day <= datetime.date.today():
    found = search_one_day('Soros', day)
    collected[day.isoformat()] = found
    print(f'{day} -> {len(found)} articles')
    day += one_day

assert len(collected) == 8
assert all(len(v) <= 20 for v in collected.values())

# the full loop, for reference - do NOT run this in a classroom of 20:
# day = datetime.date(1999, 1, 1)
# while day <= datetime.date.today():
#     collected[day.isoformat()] = search_one_day('Soros', day)
#     time.sleep(0.5)
#     day += one_day

### Act II: the main articles from telex.hu

On the front page each article's headline link carries the `item__title` class. The article body lives in `.article-html-content`.

In [ ]:
url = 'http://telex.hu'
telex_response = requests.get(url, headers=get_header(USER_AGENTS))
telex_response.raise_for_status()
telex_soup = BeautifulSoup(telex_response.content, 'html.parser')

TELEX_BASE = 'https://telex.hu'


def get_article(sub_url):
    """Return the title, text, url and pictures of one telex article."""
    full_url = sub_url if sub_url.startswith('http') else TELEX_BASE + sub_url
    response = requests.get(full_url, headers=get_header(USER_AGENTS))
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')

    body = soup.select_one('.article-html-content')
    headline = soup.select_one('h1')

    return {
        'url': full_url,
        'title': headline.get_text().strip() if headline else None,
        'text': '\n'.join(p.get_text().strip()
                          for p in (body.select('p') if body else [])),
        'pictures': [img.get('src') for img in (body.select('img') if body else [])],
    }


def get_main_articles(soup, limit=3):
    links = [a.get('href') for a in soup.select('a.item__title')]
    # the same article can be linked twice (image + headline), so de-duplicate
    unique = list(dict.fromkeys(links))
    return [get_article(link) for link in unique[:limit]]


articles = get_main_articles(telex_soup, limit=3)
for article in articles:
    print(f"{article['title'][:64]}")
    print(f"   {len(article['text'])} chars, {len(article['pictures'])} pictures")

assert len(articles) == 3
assert all(a['title'] and a['url'].startswith('https://telex.hu') for a in articles)
assert any(len(a['text']) > 200 for a in articles)

### Exercise 2. What is on sale on Steam right now?

Everything is on one page and it is plain server-rendered html - no javascript, no API key. The prices come back as strings like `'11,99€'`, so they have to be cleaned before they can be added up. That cleaning step is most of the work.

In [ ]:
STEAM_SEARCH = 'https://store.steampowered.com/search/'


def to_number(price_text):
    """'11,99€' -> 11.99 ; returns None for 'Free' and friends."""
    digits = ''.join(char for char in price_text if char.isdigit() or char in ',.')
    digits = digits.replace('.', '').replace(',', '.')
    try:
        return float(digits)
    except ValueError:
        return None


def parse_specials(soup):
    games = []
    for row in soup.select('a.search_result_row'):
        discount = row.select_one('.discount_pct')
        original = row.select_one('.discount_original_price')
        final = row.select_one('.discount_final_price')
        review = row.select_one('.search_review_summary')

        games.append({
            'title': row.select_one('.title').get_text().strip(),
            'discount': discount.get_text().strip() if discount else None,
            'original': to_number(original.get_text()) if original else None,
            'price': to_number(final.get_text()) if final else None,
            'review': (review.get('data-tooltip-html', '').split('<br>')[0]
                       if review else None),
            'url': row.get('href'),
        })
    return games


response = requests.get(STEAM_SEARCH, params={'specials': 1},
                        headers=get_header(USER_AGENTS))
response.raise_for_status()
specials = parse_specials(BeautifulSoup(response.content, 'html.parser'))

print(f'{len(specials)} games on sale\n')
for game in specials[:8]:
    print(f"  {game['title'][:38]:40s} {game['discount'] or '':>6s} "
          f"{game['original'] or 0:>7.2f} -> {game['price'] or 0:>7.2f}")

assert len(specials) == 50, len(specials)
assert all(g['title'] for g in specials)
priced = [g for g in specials if g['price'] is not None and g['original'] is not None]
assert len(priced) > 40
assert all(g['price'] <= g['original'] for g in priced)

### Exercise 2 extras

In [ ]:
priced = [g for g in specials if g['price'] is not None and g['original'] is not None]

sale_total = sum(g['price'] for g in priced)
full_total = sum(g['original'] for g in priced)

deepest = max(priced, key=lambda g: int(g['discount'].strip('-%')))
biggest_saving = max(priced, key=lambda g: g['original'] - g['price'])

print(f'buying all {len(priced)} games on sale : {sale_total:8.2f}')
print(f'the same list at full price          : {full_total:8.2f}')
print(f'total saving                         : {full_total - sale_total:8.2f} '
      f'({(1 - sale_total / full_total):.1%})')
print(f"\ndeepest discount : {deepest['title'][:40]} ({deepest['discount']})")
print(f"biggest saving   : {biggest_saving['title'][:40]} "
      f"({biggest_saving['original'] - biggest_saving['price']:.2f})")

assert sale_total < full_total
# deepest percentage and biggest absolute saving are usually different games -
# that is the point of asking for both
print('\nsame game for both?', deepest['title'] == biggest_saving['title'])

### Exercise 3. Functionize

In [ ]:
def check_price(game):
    """Look up the current price of one game by name."""
    response = requests.get(STEAM_SEARCH, params={'term': game},
                            headers=get_header(USER_AGENTS))
    response.raise_for_status()
    rows = BeautifulSoup(response.content, 'html.parser').select('a.search_result_row')
    if not rows:
        return None

    row = rows[0]
    price = row.select_one('.discount_final_price') or row.select_one('.search_price')
    return {
        'title': row.select_one('.title').get_text().strip(),
        'price': to_number(price.get_text()) if price else None,
        'url': row.get('href'),
    }


def get_specials(pages=1):
    """Collect the games on sale across `pages` pages of results."""
    games = []
    for page in range(1, pages + 1):
        response = requests.get(STEAM_SEARCH, params={'specials': 1, 'page': page},
                                headers=get_header(USER_AGENTS))
        response.raise_for_status()
        games.extend(parse_specials(BeautifulSoup(response.content, 'html.parser')))
    return games


def main():
    games = get_specials(pages=2)
    priced = [g for g in games if g['price'] is not None and g['original'] is not None]
    print(f'{len(games)} games on sale')
    print(f'total at sale price: {sum(g["price"] for g in priced):.2f}')
    print(f'total saving      : '
          f'{sum(g["original"] - g["price"] for g in priced):.2f}')
    return games


hl2 = check_price('half-life 2')
print(hl2)
assert hl2 and 'Half-Life' in hl2['title'] and hl2['price'] > 0

# A real quirk worth showing the students: with no `page` parameter Steam
# returns 50 rows, but `page=1` returns only 25. So two pages == 50 games,
# and they are *different* 50 from the single-request version above.
one_page = get_specials(pages=1)
two_pages = get_specials(pages=2)
print(f'\npages=1 -> {len(one_page)} games, pages=2 -> {len(two_pages)} games')

assert len(one_page) == 25, len(one_page)
assert len(two_pages) == 50, len(two_pages)
assert len({g['title'] for g in two_pages}) == 50, 'the two pages should not overlap'

In [ ]:
main() and None   # `and None` just keeps the long list out of the output

### Final Act: `articles.py`

On a telex article page the topic tags are the `<a>` elements with the `tag` class. The script is written out below so this notebook can import it, exactly as the students will.

In [ ]:
%%writefile articles.py
# encoding: utf-8
import random

import requests
from bs4 import BeautifulSoup

BASE_URL = 'https://telex.hu'

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36',
    'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:130.0) Gecko/20100101 Firefox/130.0',
]


def get_header(agents=USER_AGENTS):
    """A browser-looking User-agent header."""
    return {'User-agent': random.choice(agents)}


def get_article_tags(telex_article_suburl):
    """Return the topic tags of one telex article, as a list of strings.

    Arguments:
        telex_article_suburl: the part of the url after telex.hu, e.g.
            '/belfold/2026/09/04/some-article'. An empty string gives the
            front page, which has no tags.
    """
    url = BASE_URL + telex_article_suburl
    response = requests.get(url, headers=get_header())
    response.raise_for_status()

    soup = BeautifulSoup(response.content, 'html.parser')
    return [tag.get_text().strip() for tag in soup.select('a.tag')]


if __name__ == '__main__':
    print(get_article_tags(''))

In [ ]:
import articles

# the front page has no article tags
assert articles.get_article_tags('') == []
assert articles.BASE_URL == 'https://telex.hu'

# a real article does
sub_url = [a.get('href') for a in telex_soup.select('a.item__title')][0]
tags = articles.get_article_tags(sub_url)
print(sub_url)
print('tags:', tags)

assert isinstance(tags, list) and len(tags) >= 1
assert all(isinstance(tag, str) and tag for tag in tags)

In [ ]:
# tidy up
import os

if os.path.exists('articles.py'):
    os.remove('articles.py')
print('cleaned up')